In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import os
import openpyxl
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.drawing.image import Image
from openpyxl.styles import Font, PatternFill
import matplotlib.pyplot as plt
import warnings
import random
import time

warnings.filterwarnings('ignore')

# -----------------------------------------------------------------------------
# 配置設定 (Configuration)
# -----------------------------------------------------------------------------
DATA_FILE = 'cleaned_stock_data1.xlsx'
OUTPUT_FILE = 'strategy_results.xlsx'
INITIAL_CAPITAL = 200_000_000  # 初始資金 2億
MAX_HOLDINGS = 10              # 最大持股數
COST_BUY = 0.001425            # 買入手續費
COST_SELL = 0.001425 + 0.003   # 賣出手續費 + 證交稅

# -----------------------------------------------------------------------------
# 資料載入類別 (DataLoader)
# -----------------------------------------------------------------------------
class DataLoader:
    def __init__(self, filepath):
        self.filepath = filepath
        self.close_prices = None
        self.dates = None
        self.tickers = None
        self.close_np = None
        # 使用字典儲存不同視窗大小的滾動統計數據 (Numpy Array)
        self.rolling_means_np = {}
        self.rolling_stds_np = {}
        self.rolling_maxs_np = {}
        self.rolling_vol_np = {}

    def load_data(self):
        """讀取 Excel 檔案並進行前處理"""
        print("正在載入數據...")
        try:
            xl = pd.ExcelFile(self.filepath)
            df_p = xl.parse('P')
            df_p['Date'] = pd.to_datetime(df_p['Date'])
            df_p.set_index('Date', inplace=True)
            # 填補缺失值 (Forward Fill)
            df_p.ffill(inplace=True)
            
            self.dates = df_p.index
            self.tickers = df_p.columns
            self.close_prices = df_p
            # 轉換為 Numpy 陣列以加速運算
            self.close_np = df_p.values
            print(f"數據載入完成: {len(self.dates)} 天, {len(self.tickers)} 檔股票。")
        except Exception as e:
            print(f"載入數據時發生錯誤: {e}")
            raise

    def precalculate_rolling_stats(self, min_w=10, max_w=100, vol_windows=range(20, 60, 5)):
        """預先計算所有可能的滾動統計量 (Mean, Std, Max) 以優化回測速度"""
        print("正在預先計算滾動統計數據...")
        for w in range(min_w, max_w + 1):
            # 預先計算 Mean, Std, Max
            self.rolling_means_np[w] = self.close_prices.rolling(window=w).mean().values
            self.rolling_stds_np[w] = self.close_prices.rolling(window=w).std().values
            self.rolling_maxs_np[w] = self.close_prices.rolling(window=w).max().values
            
        for w in vol_windows:
            # 預先計算波動率 (Standard Deviation)
            self.rolling_vol_np[w] = self.close_prices.rolling(window=w).std().values
        print("預計算完成。")

# -----------------------------------------------------------------------------
# 回測引擎類別 (BacktestEngine)
# -----------------------------------------------------------------------------
class BacktestEngine:
    def __init__(self, data_loader, params):
        self.data = data_loader
        self.params = params
        self.vol_window = int(params['vol_window'])
        self.min_lookback = int(params['min_lookback'])
        self.max_lookback = int(params['max_lookback'])
        self.bollinger_k = params['bollinger_k']
        self.equity = INITIAL_CAPITAL
        self.cash = INITIAL_CAPITAL
        self.holdings = {} 
        self.closed_trades = []
        self.equity_curve = []
        self.n_stocks = len(self.data.tickers)
        self.current_lookback = np.full(self.n_stocks, self.min_lookback, dtype=int)
        
    def run(self):
        close_np = self.data.close_np
        dates = self.data.dates
        tickers = self.data.tickers
        n_days = close_np.shape[0]
        start_idx = max(self.max_lookback, self.vol_window) + 1
        vol_data = self.data.rolling_vol_np[self.vol_window]
        for i in range(start_idx, n_days):
            idx_T = i - 1
            close_T = close_np[idx_T]
            with np.errstate(divide='ignore', invalid='ignore'):
                delta_vol = (vol_data[idx_T] - vol_data[i - 2]) / vol_data[idx_T]
            delta_vol = np.nan_to_num(delta_vol)
            new_lb = self.current_lookback * (1.0 + delta_vol)
            # Corrected line: cast to int before clipping
            rounded_lb = np.round(new_lb).astype(int)
            np.clip(rounded_lb, self.min_lookback, self.max_lookback, out=self.current_lookback)
            buy_signals = {}
            # sell_indices = [] # 移除原本的賣出索引
            unique_lbs = np.unique(self.current_lookback)
            for lb in unique_lbs:
                mask = (self.current_lookback == lb)
                roll_mean = self.data.rolling_means_np[lb][idx_T][mask]
                roll_std = self.data.rolling_stds_np[lb][idx_T][mask]
                highest_close = self.data.rolling_maxs_np[lb][idx_T][mask]
                upper_bb = roll_mean + self.bollinger_k * roll_std
                prices = close_T[mask]
                cond_buy = (prices > upper_bb) & (prices >= highest_close)
                # cond_sell = (prices < roll_mean) # 移除原本的賣出條件
                global_indices = np.flatnonzero(mask)
                if np.any(cond_buy):
                    for idx in np.flatnonzero(cond_buy):
                        s_idx = global_indices[idx]
                        reason = f"進場收盤價 {prices[idx]:.2f} > 布林上通道值 {upper_bb[idx]:.2f} 且 收盤價創下過去 {lb} 天的新高"
                        buy_signals[s_idx] = reason
                # 移除原本的 sell_indices.extend
            exec_prices = close_np[i]
            current_date = dates[i]
            
            # 新的出場邏輯
            for s_idx in list(self.holdings.keys()):
                holding = self.holdings[s_idx]
                entry_price = holding['entry_price']
                entry_idx = holding['entry_idx']
                
                # 條件 1: 收盤價 < 20MA
                ma20_T = self.data.rolling_means_np[20][idx_T][s_idx]
                cond_sell_ma = close_T[s_idx] < ma20_T
                
                # 條件 2: 進場後 10 個交易日內跌破進場點 10%
                cond_stop_loss = (idx_T - entry_idx < 10) and (close_T[s_idx] < entry_price * 0.9)
                
                if cond_sell_ma or cond_stop_loss:
                    price = exec_prices[s_idx]
                    if np.isnan(price): continue
                    
                    if cond_stop_loss:
                        exit_reason = f"停損: 進場 {idx_T - entry_idx + 1} 日內跌破進場點 10% (進場價: {entry_price:.2f}, 目前價: {close_T[s_idx]:.2f})"
                    else:
                        exit_reason = f"出場: 收盤價 {close_T[s_idx]:.2f} 跌破 20 日均線 {ma20_T:.2f}"
                        
                    net_revenue = (holding['qty'] * price) * (1 - COST_SELL)
                    self.cash += net_revenue
                    self.closed_trades.append({
                        'Ticker': tickers[s_idx],
                        'Buy Date': holding['entry_date'],
                        'Buy Price': holding['entry_price'],
                        'Sell Date': current_date,
                        'Sell Price': price,
                        'Shares': holding['qty'],
                        'PnL': net_revenue - holding['cost'],
                        'Return': (net_revenue / holding['cost']) - 1,
                        'Lookback': holding['lookback'],
                        'Entry Reason': holding.get('entry_reason', ''),
                        'Exit Reason': exit_reason
                    })
                    del self.holdings[s_idx]
            
            if len(self.holdings) < MAX_HOLDINGS:
                holdings_val = sum(info['qty'] * exec_prices[s_idx] for s_idx, info in self.holdings.items() if not np.isnan(exec_prices[s_idx]))
                curr_equity = self.cash + holdings_val
                target_size = curr_equity / MAX_HOLDINGS
                for s_idx, reason in buy_signals.items():
                    if len(self.holdings) >= MAX_HOLDINGS: break
                    if s_idx in self.holdings: continue
                    price = exec_prices[s_idx]
                    if np.isnan(price) or price <= 0: continue
                    cost_basis = price * (1 + COST_BUY)
                    shares = min(int(self.cash / cost_basis), int(target_size / cost_basis))
                    if shares > 0:
                        total_cost = shares * cost_basis
                        self.cash -= total_cost
                        self.holdings[s_idx] = {
                            'qty': shares,
                            'cost': total_cost,
                            'entry_date': current_date,
                            'entry_price': price,
                            'entry_idx': i,  # 記錄進場時的索引
                            'lookback': int(self.current_lookback[s_idx]),
                            'entry_reason': reason
                        }
            holdings_val = sum(info['qty'] * exec_prices[s_idx] for s_idx, info in self.holdings.items() if not np.isnan(exec_prices[s_idx]))
            self.equity_curve.append({'Date': current_date,'Equity': self.cash + holdings_val,'Cash': self.cash,'Holdings_Count': len(self.holdings),'Holdings': ", ".join(f"{tickers[s_idx]}:{info['qty']}" for s_idx, info in self.holdings.items())})

    def get_metrics(self):
        """計算策略績效指標"""
        if not self.equity_curve:
            return {'Calmar': -999, 'CAGR': 0, 'MaxDD': 0}
        
        df = pd.DataFrame(self.equity_curve).set_index('Date')
        
        # 計算年化報酬率 (CAGR)
        days = (df.index[-1] - df.index[0]).days
        years = days / 365.25
        if years == 0: years = 0.001
        
        total_ret = df['Equity'].iloc[-1] / INITIAL_CAPITAL - 1
        cagr = (1 + total_ret) ** (1/years) - 1
        
        # 計算最大回撤 (Max Drawdown)
        roll_max = df['Equity'].cummax()
        drawdown = (df['Equity'] - roll_max) / roll_max
        max_dd = drawdown.min()
        
        # 計算 Calmar Ratio
        if max_dd == 0: calmar = 0
        else: calmar = cagr / abs(max_dd)
        
        # 計算勝率 (Win Rate)
        win_rate = 0.0
        if self.closed_trades:
            wins = sum(1 for t in self.closed_trades if t['PnL'] > 0)
            win_rate = wins / len(self.closed_trades)

        return {
            'Calmar': calmar,
            'CAGR': cagr,
            'MaxDD': max_dd,
            'WinRate': win_rate,
            'Final_Equity': df['Equity'].iloc[-1],
            'History': df,
            'Trades': self.closed_trades
        }

# -----------------------------------------------------------------------------
# 結果輸出函式 (Excel Generator)
# -----------------------------------------------------------------------------
def save_excel(metrics, params, filename=OUTPUT_FILE):
    print(f"正在儲存結果至 {filename}...")
    wb = Workbook()
    ws_sum = wb.active
    ws_sum.title = "Summary"
    ws_sum.append(["Metric", "Value"])
    ws_sum.append(["CAGR (年化報酬率)", f"{metrics['CAGR']:.2%}"])
    ws_sum.append(["Max Drawdown (最大回撤)", f"{metrics['MaxDD']:.2%}"])
    ws_sum.append(["Calmar Ratio", f"{metrics['Calmar']:.2f}"])
    ws_sum.append(["Win Rate (勝率)", f"{metrics['WinRate']:.2%}"])
    ws_sum.append(["Final Equity (最終權益)", f"{metrics['Final_Equity']:,.0f}"])
    ws_sum.append(["Initial Capital (初始資金)", f"{INITIAL_CAPITAL:,.0f}"])
    ws_sum.append([])
    ws_sum.append(["Parameters (使用參數)", ""])
    for k, v in params.items():
        ws_sum.append([k, v])
    ws_trades = wb.create_sheet("Trades")
    headers = ["Ticker", "Buy Date", "Buy Price", "Sell Date", "Sell Price", "Shares", "PnL", "Return", "Lookback", "Entry Reason", "Exit Reason"]
    ws_trades.append(headers)
    for t in metrics['Trades']:
        ws_trades.append([t['Ticker'], t['Buy Date'], t['Buy Price'], t['Sell Date'], t['Sell Price'], t['Shares'], t['PnL'], f"{t['Return']:.2%}", t.get('Lookback', ''), t.get('Entry Reason', ''), t.get('Exit Reason', '')])
    ws_curve = wb.create_sheet("Equity_Curve")
    df_hist = metrics['History']
    df_hist.reset_index(inplace=True)
    roll_max = df_hist['Equity'].cummax()
    df_hist['Drawdown'] = (df_hist['Equity'] - roll_max) / roll_max
    for r in dataframe_to_rows(df_hist, index=False, header=True):
        ws_curve.append(r)
    plt.figure(figsize=(10, 6))
    plt.plot(df_hist['Date'], df_hist['Equity'], label='Equity')
    plt.fill_between(df_hist['Date'], df_hist['Equity'], INITIAL_CAPITAL, alpha=0.1)
    plt.title("Strategy Equity Curve")
    plt.xlabel("Date")
    plt.ylabel("Equity (TWD)")
    plt.legend()
    plt.grid(True)
    img_path = "equity_curve.png"
    plt.savefig(img_path)
    plt.close()
    img = Image(img_path)
    ws_curve.add_image(img, "H2")
    ws_hold = wb.create_sheet("Equity_Hold")
    ws_hold.append(["Date", "Holdings_Count", "Details"])
    for index, row in df_hist.iterrows():
        ws_hold.append([row['Date'], row['Holdings_Count'], row['Holdings']])
    ws_daily = wb.create_sheet("Daily_Record")
    ws_daily.append(["Date", "Equity", "Cash", "Holdings_Count"])
    for index, row in df_hist.iterrows():
        ws_daily.append([row['Date'], row['Equity'], row['Cash'], row['Holdings_Count']])
    wb.save(filename)
    print("Excel 檔案儲存完成。")

# -----------------------------------------------------------------------------
# 主程式進入點 (Main)
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    loader = DataLoader(DATA_FILE)
    loader.load_data()
    loader.precalculate_rolling_stats()
    fixed_params = {
        'vol_window': 40,
        'min_lookback': 10,
        'max_lookback': 40,
        'bollinger_k': 3.4
    }
    print("使用固定參數組合:", fixed_params)
    print("執行最終回測...")
    final_engine = BacktestEngine(loader, fixed_params)
    final_engine.run()
    final_metrics = final_engine.get_metrics()
    save_excel(final_metrics, fixed_params)
